In [ ]:
# MEDIA_DIR -> media/ (gitignored: Wigner animations), resolved from the
# installed package so this notebook runs from any working directory.
from hhb.paths import MEDIA_DIR

MEDIA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
"""
Harmonic-balance solver for the driven-Kerr moment equations
    mu_dot        = -i*eps/2*(1+exp(2i wd t)) - (i*Delta + kappa/2 - 2i*chi) mu
                    - i*chi*(mu^2 mu* + 2 mu sigma2 + mu* sigmatilde2)
    sigma2_dot    = kappa*(1-sigma2) - 2*chi*Im[mu*^2 sigmatilde2]
    sigmatilde2_dot = -(2i*Delta+kappa-5i*chi+4i*chi*|mu|^2+6i*chi*sigma2)*sigmatilde2
                      + i*chi*mu^2*(1-2*sigma2)

Adapted from hill_method.py (Detroux et al. 2015 harmonic-balance / AFT structure),
but for a FIRST-ORDER complex ODE system instead of a second-order mechanical one.

State packed as n=5 real DOFs: y = [Re(mu), Im(mu), sigma2, Re(sigmatilde2), Im(sigmatilde2)]
"""

import numpy as np
import sympy as sp
from scipy.sparse import lil_matrix
from scipy.optimize import root
from dataclasses import dataclass


@dataclass
class KerrParams:
    Delta: float
    chi: float
    kappa: float
    eps: float
    omega_d: float


@dataclass
class HBSettingsMoments:
    N_H: int = 8                # keep even harmonics 0..N_H (odd ones stay zero by parity)
    samples_per_harmonic: int = 64
    nu: int = 1
    n: int = 5                   # Re(mu), Im(mu), sigma2, Re(sigmatilde2), Im(sigmatilde2)

    @property
    def N(self):
        return self.samples_per_harmonic * self.N_H


# ----------------------------------------------------------------------
# Symbolic derivation of the nonlinear RHS + its Jacobian (done ONCE,
# lambdified for fast numeric evaluation). This avoids re-deriving the
# 5x5 Jacobian by hand -- exactly the kind of algebra that has produced
# sign/index errors earlier in this derivation.
# ----------------------------------------------------------------------
def _build_symbolic_rhs():
    m1, m2, s, t1, t2, chi, kappa, Delta = sp.symbols(
        'm1 m2 s t1 t2 chi kappa Delta', real=True
    )
    mu = m1 + sp.I * m2
    mu_c = m1 - sp.I * m2
    sig = s
    sigt = t1 + sp.I * t2

    dmu = -(sp.I * Delta + kappa / 2 - 2 * sp.I * chi) * mu \
          - sp.I * chi * (mu**2 * mu_c + 2 * mu * sig + mu_c * sigt)
    ds = kappa * (-sig) - 2 * chi * sp.im(mu_c**2 * sigt)
    dsigt = -(2 * sp.I * Delta + kappa - 5 * sp.I * chi
              + 4 * sp.I * chi * mu * mu_c + 6 * sp.I * chi * sig) * sigt \
             + sp.I * chi * mu**2 * (1 - 2 * sig)

    F = sp.Matrix([
        sp.re(dmu), sp.im(dmu), sp.expand(ds), sp.re(dsigt), sp.im(dsigt)
    ])
    y = sp.Matrix([m1, m2, s, t1, t2])

    # Lin, built EXACTLY as in MomentHillMethod._build_Lin -- must match so that
    # N = F_full - Lin @ y keeps only the genuinely nonlinear (state-bilinear/
    # cubic) part. Otherwise the linear part gets double-counted between L
    # (harmonic-balance operator) and b_nl (AFT nonlinear term) -- exactly the
    # bug that produced a spurious factor in the chi=0 test.
    a1, b1 = -kappa / 2, -(Delta - 2 * chi)
    a2, b2 = -kappa, (5 * chi - 2 * Delta)
    Lin_sym = sp.Matrix([
        [a1, -b1, 0, 0, 0],
        [b1,  a1, 0, 0, 0],
        [0,   0, -kappa, 0, 0],
        [0,   0, 0, a2, -b2],
        [0,   0, 0, b2,  a2],
    ])

    N = sp.simplify(F - Lin_sym * y)
    J = N.jacobian(y)

    params = (chi, kappa, Delta)
    args = (m1, m2, s, t1, t2) + params
    N_funcs = [sp.lambdify(args, N[i], modules='numpy') for i in range(5)]
    J_funcs = [[sp.lambdify(args, J[i, j], modules='numpy') for j in range(5)]
               for i in range(5)]
    return N_funcs, J_funcs


_F_SYM, _J_SYM = _build_symbolic_rhs()


class MomentHillMethod:
    def __init__(self, kerr: KerrParams, settings: HBSettingsMoments):
        self.p = kerr
        self.s = settings
        self.n = settings.n
        self.N_H = settings.N_H
        self.N = settings.N
        self.dim = self.n * (2 * self.N_H + 1)

        # one TRUE fundamental period of the HB basis (frequencies k*omega_d/nu):
        # T_fund = 2*pi*nu/omega_d in physical time (CLAUDE.md bug #5).
        self.T_fund = 2 * np.pi * self.s.nu / self.p.omega_d
        self.tau_j = np.linspace(0, self.T_fund, self.N, endpoint=False)

        self.Lin = self._build_Lin()
        self.L = self.build_L()
        self.Gamma = self.build_Gamma()
        self.Gamma_pinv = np.linalg.pinv(self.Gamma, rcond=1e-12)
        self.b_ext = self.build_b_ext()

    # ---------- linear (constant-coefficient) part of the ODE --------
    def _build_Lin(self):
        p = self.p
        a1, b1 = -p.kappa / 2, -(p.Delta - 2 * p.chi)     # mu block
        a2, b2 = -p.kappa, (5 * p.chi - 2 * p.Delta)      # sigmatilde2 block
        Lin = np.array([
            [a1, -b1, 0, 0, 0],
            [b1,  a1, 0, 0, 0],
            [0,   0, -p.kappa, 0, 0],
            [0,   0, 0, a2, -b2],
            [0,   0, 0, b2,  a2],
        ])
        return Lin

    # ---------- first-order harmonic-balance linear operator ----------
    def build_L(self):
        """
        L such that: (d/dt of Fourier series) - Lin*(coeffs) <-> L @ z
        Ordering [c0, s1, c1, ..., s_NH, c_NH], each block in R^n (n=5).
        """
        n, N_H, nu = self.n, self.N_H, self.s.nu
        omega = self.p.omega_d
        dim = self.dim
        L = np.zeros((dim, dim))

        def idx_c0():  return 0
        def idx_sk(k): return 1 + 2 * (k - 1)
        def idx_ck(k): return 2 + 2 * (k - 1)

        i0 = idx_c0()
        L[i0 * n:(i0 + 1) * n, i0 * n:(i0 + 1) * n] = -self.Lin

        for k in range(1, N_H + 1):
            freq = k * omega / nu
            is_, ic_ = idx_sk(k), idx_ck(k)
            # derivative block (paper's nabla_k) tensor I_n, minus Lin on diagonal
            L[is_ * n:(is_ + 1) * n, is_ * n:(is_ + 1) * n] = -self.Lin
            L[is_ * n:(is_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, is_ * n:(is_ + 1) * n] = freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -self.Lin

        return L

    # ---------- AFT machinery (identical structure to hill_method.py) --
    def build_Gamma(self):
        t, N_H, n, nu = self.tau_j, self.N_H, self.n, self.s.nu
        omega = self.p.omega_d
        k = np.arange(1, N_H + 1)
        S = np.sin(np.outer(t, k * omega / nu))
        C = np.cos(np.outer(t, k * omega / nu))
        Phi = np.empty((t.size, 2 * N_H + 1))
        Phi[:, 0] = 1.0 / np.sqrt(2.0)
        Phi[:, 1::2] = S
        Phi[:, 2::2] = C
        return np.kron(Phi, np.eye(n))

    def x_tilde_to_X(self, xt):
        return xt.reshape(self.N, self.n).T   # (n, N)

    def X_to_x_tilde(self, X):
        return X.T.reshape(-1)

    def N_time(self, X):
        """Nonlinear part of the RHS, evaluated pointwise in time. X: (5,N)."""
        m1, m2, s, t1, t2 = X
        Ncol = X.shape[1]
        out = np.empty((5, Ncol))
        for i in range(5):
            val = _F_SYM[i](m1, m2, s, t1, t2, self.p.chi, self.p.kappa, self.p.Delta)
            out[i, :] = np.broadcast_to(val, (Ncol,))
        return out

    def dN_dX_blocks(self, X):
        m1, m2, s, t1, t2 = X
        Ncol = X.shape[1]
        Jarr = np.zeros((5, 5, Ncol))
        for i in range(5):
            for j in range(5):
                val = _J_SYM[i][j](m1, m2, s, t1, t2, self.p.chi, self.p.kappa, self.p.Delta)
                Jarr[i, j, :] = np.broadcast_to(val, (Ncol,))
        return [Jarr[:, :, k] for k in range(Ncol)]

    def build_dNtilde_dx_tilde(self, X):
        blocks = self.dN_dX_blocks(X)
        Jbig = lil_matrix((self.n * self.N, self.n * self.N))
        for j, Jj in enumerate(blocks):
            rows = slice(j * self.n, (j + 1) * self.n)
            Jbig[rows, rows] = Jj
        return Jbig.tocsr()

    def b_nl(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Ftime = self.N_time(X)
        return self.Gamma_pinv @ self.X_to_x_tilde(Ftime)

    def db_dz(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Jbig = self.build_dNtilde_dx_tilde(X)
        return self.Gamma_pinv @ (Jbig @ self.Gamma)

    # ---------- forcing (kappa constant + eps drive at k=0,2) ----------
    def build_b_ext(self):
        n, N_H = self.n, self.N_H
        b = np.zeros(self.dim)

        def idx_c0():  return 0
        def idx_sk(k): return 1 + 2 * (k - 1)
        def idx_ck(k): return 2 + 2 * (k - 1)

        eps = self.p.eps
        i0 = idx_c0()
        # kappa*(sigma2 eq constant "+kappa"): physical DC value kappa -> c0 = kappa*sqrt2
        b[i0 * n + 2] += self.p.kappa * np.sqrt(2.0)
        # mu-eq forcing: -i*eps/2*(1 + exp(2i wd t))
        #   constant part -i*eps/2 -> Im(mu)-forcing at DC: c0 = -eps/2 * sqrt2
        b[i0 * n + 1] += -eps / 2 * np.sqrt(2.0)
        if N_H >= 2:
            is2, ic2 = idx_sk(2), idx_ck(2)
            # -i*eps/2*exp(2i wd t) = eps/2*sin(2 wd t) - i*eps/2*cos(2 wd t)
            b[is2 * n + 0] += eps / 2.0        # Re(mu) forcing, sin(2 wd t)
            b[ic2 * n + 1] += -eps / 2.0       # Im(mu) forcing, cos(2 wd t)
        return b

    # ---------- residual / Jacobian / solve --------------------------
    def residual(self, z):
        return self.L @ z - self.b_nl(z) - self.b_ext

    def jacobian(self, z):
        return self.L - self.db_dz(z)

    def solve(self, z0, **kwargs):
        sol = root(self.residual, z0, jac=self.jacobian, method='hybr', **kwargs)
        if not sol.success:
            raise RuntimeError(f"HB solve did not converge: {sol.message}")
        return sol.x

In [ ]:
"""
Example usage of MomentHillMethod, plus an independent check against direct
time integration of the same moment ODEs (the same kind of cross-check used
throughout the derivation: HB should reproduce the periodic steady state
reached by long-time integration).
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

#from hill_method_moments import KerrParams, HBSettingsMoments, MomentHillMethod

Delta = 1.0
chi = 0.0
kappa = 1.0
eps = 3
omega_d=1.3

# ----------------------------------------------------------------------
# 1) Choose physical parameters and solve with harmonic balance
# ----------------------------------------------------------------------
kerr = KerrParams(
    Delta=Delta,      # detuning
    chi=chi,       # Kerr strength
    kappa=kappa,      # decay rate
    eps=eps,        # drive amplitude
    omega_d=omega_d,    # drive frequency
)
settings = HBSettingsMoments(N_H=8, samples_per_harmonic=48)

hb = MomentHillMethod(kerr, settings)

# initial guess: coherent-ish, sigma2 near 1 (vacuum-noise level), no squeezing
z0 = np.zeros(hb.dim)
z0[2] = np.sqrt(2) * 1.0   # DC sigma2 guess

z_sol = hb.solve(z0)
print(f"[HB] residual norm at solution: {np.linalg.norm(hb.residual(z_sol)):.3e}")

# reconstruct the periodic steady state over one drive period from the HB solution
X_hb = hb.x_tilde_to_X(hb.Gamma @ z_sol)          # shape (5, N)
t_hb = hb.tau_j                                    # already physical time (one period)
mu_hb = X_hb[0] + 1j * X_hb[1]
sig_hb = X_hb[2]
sigt_hb = X_hb[3] + 1j * X_hb[4]


# ----------------------------------------------------------------------
# 2) Independent check: integrate the ODEs directly in time
# ----------------------------------------------------------------------
def rhs(t, y, p: KerrParams):
    mu = y[0] + 1j * y[1]
    s = y[2]
    sigt = y[3] + 1j * y[4]
    eps_t = p.eps / 2 * (1 + np.exp(2j * p.omega_d * t))

    dmu = (-1j * eps_t
           - (1j * p.Delta + p.kappa / 2 - 2j * p.chi) * mu
           - 1j * p.chi * (mu**2 * np.conj(mu) + 2 * mu * s + np.conj(mu) * sigt))
    ds = p.kappa * (1 - s) - 2 * p.chi * np.imag(np.conj(mu)**2 * sigt)
    dsigt = (-(2j * p.Delta + p.kappa - 5j * p.chi
               + 4j * p.chi * abs(mu)**2 + 6j * p.chi * s) * sigt
             + 1j * p.chi * mu**2 * (1 - 2 * s))
    return [dmu.real, dmu.imag, ds, dsigt.real, dsigt.imag]


T = 2 * np.pi / (kerr.omega_d)
n_periods_transient = 600     # long enough for kappa-controlled transients to decay
y0 = [0.0, 0.0, 1.0, 0.0, 0.0]

sol_ti = solve_ivp(
    rhs, [0, n_periods_transient * T], y0, args=(kerr,),
    max_step=T / 300, dense_output=True, rtol=1e-10, atol=1e-12,
)

# sample the last period only (steady state)
t_last = np.linspace(sol_ti.t[-1] - T, sol_ti.t[-1], 400)
Y_ss = sol_ti.sol(t_last)
mu_ti = Y_ss[0] + 1j * Y_ss[1]
sig_ti = Y_ss[2]
sigt_ti = Y_ss[3] + 1j * Y_ss[4]
t_ti_phase = (t_last - t_last[0]) % T   # phase within the period, for comparison


# ----------------------------------------------------------------------
# 3) Compare
# ----------------------------------------------------------------------
# interpolate the HB curve (evaluated on hb.tau_j grid) onto the time-integration phase grid
def interp_periodic(t_query, t_grid, values, T):
    # values may be complex; interpolate real/imag separately with periodic wrap
    t_grid_ext = np.concatenate([t_grid, t_grid[:1] + T])
    re = np.concatenate([values.real, values.real[:1]])
    im = np.concatenate([values.imag, values.imag[:1]])
    re_i = np.interp(t_query, t_grid_ext, re)
    im_i = np.interp(t_query, t_grid_ext, im)
    return re_i + 1j * im_i


mu_hb_interp = interp_periodic(t_ti_phase, t_hb, mu_hb, T)
sig_hb_interp = interp_periodic(t_ti_phase, t_hb, sig_hb.astype(complex), T).real
sigt_hb_interp = interp_periodic(t_ti_phase, t_hb, sigt_hb, T)

err_mu = np.max(np.abs(mu_hb_interp - mu_ti)) / np.max(np.abs(mu_ti))
err_sig = np.max(np.abs(sig_hb_interp - sig_ti)) / np.max(np.abs(sig_ti))
err_sigt = np.max(np.abs(sigt_hb_interp - sigt_ti)) / max(np.max(np.abs(sigt_ti)), 1e-12)

print(f"[check] relative max error, mu:          {err_mu:.3e}")
print(f"[check] relative max error, sigma^2:      {err_sig:.3e}")
print(f"[check] relative max error, sigma_tilde^2:{err_sigt:.3e}")

assert err_mu < 1e-4, "harmonic balance does not match time integration for mu!"
assert err_sig < 1e-4, "harmonic balance does not match time integration for sigma^2!"
print("HB solution matches direct time-integration steady state. Good.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

ax = axes[0, 0]
ax.plot(t_ti_phase / T, mu_ti.real, 'k-', label='time-int, Re(mu)')
ax.plot(t_hb / T, mu_hb.real, 'C0--', label='HB, Re(mu)')
ax.plot(t_ti_phase / T, mu_ti.imag, 'gray', label='time-int, Im(mu)')
ax.plot(t_hb / T, mu_hb.imag, 'C1--', label='HB, Im(mu)')
ax.set_xlabel('t / T'); ax.set_title('mu(t)'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(mu_ti.real, mu_ti.imag, 'k-', label='time-int')
ax.scatter(mu_hb.real, mu_hb.imag, c=t_hb / T, label='HB')
ax.set_xlabel(r'$\mathrm{Re}(\mu)$'); ax.set_ylabel(r'$\mathrm{Im}(\mu)$'); ax.set_title('phase-space trajectory')
ax.set_aspect('equal'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(t_ti_phase / T, sig_ti, 'k-', label='time-int')
ax.plot(t_hb / T, sig_hb, 'C0--', label='HB')
ax.set_xlabel('t / T'); ax.set_title('sigma^2(t)'); ax.legend(fontsize=8)

ax = axes[1, 1]
ax.plot(t_ti_phase / T, sigt_ti.real, 'k-', label='time-int, Re')
ax.plot(t_hb / T, sigt_hb.real, 'C0--', label='HB, Re')
ax.plot(t_ti_phase / T, sigt_ti.imag, 'gray', label='time-int, Im')
ax.plot(t_hb / T, sigt_hb.imag, 'C1--', label='HB, Im')
ax.set_xlabel('t / T'); ax.set_title('sigma_tilde^2(t)'); ax.legend(fontsize=8)

fig.tight_layout()
plt.plot()

In [ ]:
import numpy as np
import dynamiqs as dq
import jax.numpy as jnp

n_a = 20
a = dq.destroy(n_a)

H_0 = Delta*a.dag()@a + chi/2*(a.dag()@a.dag())@(a@a)
H_drive_a = dq.modulated(lambda t: eps/2*(1+jnp.exp(-2j*omega_d*t)), a)
H_drive_a_dag = dq.modulated(lambda t: eps/2*(1+jnp.exp(2j*omega_d*t)), a.dag())
H_tot = H_0 + H_drive_a + H_drive_a_dag

jump_op = [jnp.sqrt(kappa)*a]

T = 2*np.pi/(2*omega_d)          # fundamental period of the drive (2*omega_d component)
t_max = 20*T
t_save = np.linspace(0, t_max, 4_000)

psi_0 = dq.coherent(n_a, -2)
res = dq.mesolve(H_tot, jump_op, psi_0, t_save)

a_exp = np.asarray(dq.expect(a, res.states)).squeeze()   # <a>(t), complex

In [ ]:
import matplotlib.pyplot as plt

n_periods_check = 2
mask_check = t_save >= t_max - n_periods_check*T
t_check = t_save[mask_check]
a_check = a_exp[mask_check]
phase_check = (t_check - t_check.min()) % T

order = np.argsort(phase_check)
fig, axs = plt.subplots(1,2, figsize=(9,3.5))
for k in range(n_periods_check):
    sel = (t_check - t_check.min()) >= k*T - 1e-9
    sel &= (t_check - t_check.min()) < (k+1)*T + 1e-9
    axs[0].plot(phase_check[sel][np.argsort(phase_check[sel])],
                a_check[sel].real[np.argsort(phase_check[sel])], label=f'period {k}')
    axs[1].plot(phase_check[sel][np.argsort(phase_check[sel])],
                a_check[sel].imag[np.argsort(phase_check[sel])], label=f'period {k}')
axs[0].set_title('Re<a> vs phase'); axs[1].set_title('Im<a> vs phase')
for ax in axs: ax.set_xlabel('t mod T'); ax.legend()
plt.tight_layout()

In [ ]:
n_periods_use = 2
mask = t_save >= t_max - n_periods_use*T
t_use = t_save[mask]
states_use = [res.states[i] for i in np.where(mask)[0]]
# absolute-time phase: the drive phase is set by exp(2i*omega_d*t) at absolute t,
# so fold t_use itself, not (t_use - t_use.min()) whose origin is off-grid
t_mod = t_use % T

In [ ]:
def make_fourier_interp(samples, T):
    """samples: 1D array (real or complex) uniformly sampled on [0,T)."""
    N = samples.size
    coeffs = np.fft.fft(samples) / N
    freqs = np.fft.fftfreq(N, d=T/N) * 2*np.pi   # angular frequencies
    def f_at(t):
        t = np.atleast_1d(t)
        phase = np.exp(1j*np.outer(t, freqs))     # (len(t), N)
        return phase @ coeffs
    return f_at

# The HB samples live on hb.tau_j, which spans hb.T_fund = 2*pi*nu/omega_d --
# NOT the dynamiqs folding period T = pi/omega_d defined above. Build the
# interpolant with the HB period and evaluate at ABSOLUTE times t_use (the
# interpolant is T_fund-periodic, so absolute time is always safe).
mu_interp   = make_fourier_interp(mu_hb,   hb.T_fund)
sig_interp  = make_fourier_interp(sig_hb,  hb.T_fund)   # sigma^2, should come out ~real
sigt_interp = make_fourier_interp(sigt_hb, hb.T_fund)

mu_t   = mu_interp(t_use)
sig_t  = np.real(sig_interp(t_use))              # enforce realness, drop fft roundoff imag part
sigt_t = sigt_interp(t_use)

# physicality check at every phase used: with n = <da+ da> = sigma^2 - 1
# and m = <da^2>, a single-mode Gaussian is physical iff n(n+1) >= |m|^2.
# (sigma^2 >= 1 + |m| is too strict -- it flags the exact Lindblad state,
#  physical by construction, as unphysical.)
n_occ = sig_t - 1.0
viol = n_occ * (n_occ + 1.0) < np.abs(sigt_t)**2 - 1e-9
if viol.any():
    print(f"WARNING: Gaussian ansatz unphysical at {viol.sum()}/{len(viol)} sampled phases")

In [ ]:
def Q_hb_grid(X, Y, mu, sig, sigt):
    """Gaussian Q function; sig is sigma^2 = <da da^dag> (anti-normal), sigt = <da^2>.
    Normalization is 1/(pi*sqrt(det)) with det = sig^2 - |sigt|^2 (check: thermal
    state sig = nbar+1 gives 1/(pi*(nbar+1)))."""
    Z = (X + 1j*Y) - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi/np.sqrt(denom)
            * np.exp((-sig*np.abs(Z)**2 + np.real(np.conj(sigt)*Z**2)) / denom))

In [ ]:
import matplotlib.animation as animation
real_axis = np.linspace(-4, 4, 100)
imag_axis = np.linspace(-4, 4, 100)

fig, ax = plt.subplots()

# --- initial frame ---
def compute_H(idx_time):
    return Q_hb_grid(X, Y, mu_hb[idx_time], sig_hb[idx_time], sigt_hb[idx_time])

X, Y = np.meshgrid(real_axis, imag_axis)
H0   = compute_H(0)

mesh = ax.pcolormesh(real_axis, imag_axis, H0, cmap='RdBu_r')
scat = ax.scatter(mu_hb[0].real, mu_hb[0].imag, color='black', s=20)
plt.colorbar(mesh, ax=ax, label='Q')
ax.set_xlabel(r'Re$(\alpha)$')
ax.set_ylabel(r'Im$(\alpha)$')
title = ax.set_title('t = 0')

# --- update function ---
def update(idx_time):
    H = compute_H(idx_time)
    mesh.set_array(H.ravel())
    mesh.set_clim(H.min(), H.max())
    scat.set_offsets([mu_hb[idx_time].real, mu_hb[idx_time].imag])
    title.set_text(f't = {idx_time}')
    return mesh, scat, title

ani = animation.FuncAnimation(
    fig, update, frames=len(mu_hb), interval=50, blit=True
)

plt.tight_layout()
plt.show()

#Optional: save as gif or mp4
#ani.save(MEDIA_DIR / "wigner_kerr.gif", writer="pillow", fps=50)

In [ ]:
def coherent_overlaps(alpha_grid, n_fock):
    n = np.arange(n_fock)
    log_norm = -0.5*np.abs(alpha_grid)[:,None]**2 - 0.5*np.array(
        [np.sum(np.log(np.arange(1,k+1))) for k in n])[None,:]
    C = np.exp(log_norm) * alpha_grid[:,None]**n[None,:]
    return C  # (M, n_fock)

alpha_grid = (X + 1j*Y).ravel()
C = coherent_overlaps(alpha_grid, n_a)   # (M, n_a), precompute once

def Q_exact_grid(rho):
    rho = np.asarray(rho)
    Q_flat = np.real(np.einsum('mi,ij,mj->m', C.conj(), rho, C)) / np.pi
    return Q_flat.reshape(X.shape)

In [ ]:
dx = real_axis[1]-real_axis[0]
dy = imag_axis[1]-imag_axis[0]
dA = dx*dy

def hellinger(Q1, Q2):
    return np.sqrt(max(0.0, 1 - np.sum(np.sqrt(np.clip(Q1*Q2,0,None)))*dA))

distances = []
for k, rho_k in enumerate(states_use):
    Q_ex = Q_exact_grid(rho_k)
    Q_hb = Q_hb_grid(X, Y, mu_t[k], sig_t[k], sigt_t[k])
    distances.append(hellinger(Q_ex, Q_hb))

distances = np.array(distances)
order = np.argsort(t_mod)
phase_sorted = t_mod[order]
dist_sorted = distances[order]

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(phase_sorted/T, dist_sorted, '.-')
plt.xlabel('phase  t mod T / T')
plt.ylabel('Hellinger distance')
plt.title('Q_HB vs Q_exact over one drive period')
plt.tight_layout()

# time-average over one period (trapezoid on the folded, sorted phase axis, not a raw sum)
avg_dist = np.trapezoid(dist_sorted, phase_sorted) / T
max_dist = dist_sorted.max()
worst_phase = phase_sorted[np.argmax(dist_sorted)]

print(f"period-averaged Hellinger distance: {avg_dist:.4f}")
print(f"worst-case distance: {max_dist:.4f} at phase {worst_phase/T:.3f} T")